## Mount Drive


In [1]:
from google.colab import drive

drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Configure


In [2]:
from pathlib import Path

PARSEQ_DRIVE_DIR = Path("/content/drive/MyDrive/Parseq")
DRIVE_TRAINING_DIR = PARSEQ_DRIVE_DIR / "Training" / "no_val"
REPO_ZIP = PARSEQ_DRIVE_DIR / "parseq_full_data_patch_token_aware_pretrained_transfer_20260527.zip"
DATA_SOURCE_DIR = DRIVE_TRAINING_DIR / "data"
CUSTOM_CHARSET_YAML = DRIVE_TRAINING_DIR / "hybrid_no_val.yaml"
WORK_DATA_DIR = Path("/content/parseq_data")
RUNS_DIR = DRIVE_TRAINING_DIR / "runs"
PARSEQ_DIR = Path("/content/parseq")
EXTRACT_DIR = Path("/content/parseq_repo_extract")
CHARSET = "hybrid_no_val"
USE_CUSTOM_CHARSET = True
NO_VALIDATION_RUN = True
AUGMENT = True
BATCH_SIZE = 32
MAX_EPOCHS = 200
MAX_LABEL_LENGTH = 44
NUM_WORKERS = 2
LOG_EVERY_N_STEPS = 5
ENABLE_PROGRESS_BAR = "true"
PRETRAINED = "parseq-tiny"
PRETRAINED_CHARSET = "94_full"
PRETRAINED_TRANSFER_VOCAB_BY_TOKEN = True
PRETRAINED_INIT_NEW_TOKENS_FROM_SPELLING = True

print("Repo zip:", REPO_ZIP)
print("Data:", DATA_SOURCE_DIR)
print("Charset YAML:", CUSTOM_CHARSET_YAML)
print("Runs:", RUNS_DIR)


Repo zip: /content/drive/MyDrive/Parseq/parseq_full_data_patch_token_aware_pretrained_transfer_20260527.zip
Data: /content/drive/MyDrive/Parseq/Training/no_val/data
Charset YAML: /content/drive/MyDrive/Parseq/Training/no_val/hybrid_no_val.yaml
Runs: /content/drive/MyDrive/Parseq/Training/no_val/runs


## Prepare Repository


In [3]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

if not REPO_ZIP.exists():
    matches = sorted(PARSEQ_DRIVE_DIR.glob("parseq_full_data_patch_token_aware_pretrained_transfer_*.zip"))
    if not matches:
        raise FileNotFoundError(REPO_ZIP)
    REPO_ZIP = matches[-1]

if PARSEQ_DIR.exists():
    shutil.rmtree(PARSEQ_DIR)
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(REPO_ZIP, "r") as zf:
    zf.extractall(EXTRACT_DIR)

def looks_like_repo(path: Path) -> bool:
    return (path / "strhub").is_dir() and (path / "configs").is_dir() and (path / "train.py").is_file()

repo_candidates = [EXTRACT_DIR] + [p for p in EXTRACT_DIR.rglob("*") if p.is_dir()]
repo_root = next((p for p in repo_candidates if looks_like_repo(p)), None)
if repo_root is None:
    raise FileNotFoundError("PARSeq repo root not found inside zip")

shutil.copytree(repo_root, PARSEQ_DIR)

if USE_CUSTOM_CHARSET:
    if not CUSTOM_CHARSET_YAML.exists():
        raise FileNotFoundError(CUSTOM_CHARSET_YAML)
    dest = PARSEQ_DIR / "configs" / "charset" / f"{CHARSET}.yaml"
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(CUSTOM_CHARSET_YAML, dest)

required = [
    "train.py",
    "strhub/data/latex_tokenizer.py",
    "strhub/data/dataset.py",
    "strhub/data/module.py",
    "strhub/data/utils.py",
    "strhub/models/base.py",
    "strhub/models/parseq/system.py",
    "strhub/models/parseq/model.py",
    f"configs/charset/{CHARSET}.yaml",
]
missing = [rel for rel in required if not (PARSEQ_DIR / rel).exists()]
if missing:
    raise FileNotFoundError(missing)

train_text = (PARSEQ_DIR / "train.py").read_text()
if "Token-aware pretrained transfer" not in train_text:
    raise RuntimeError("token-aware pretrained transfer code was not found")

compile_targets = [PARSEQ_DIR / rel for rel in required if rel.endswith(".py")]
subprocess.run([sys.executable, "-m", "py_compile", *map(str, compile_targets)], check=True)
os.environ["PYTHONPATH"] = str(PARSEQ_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")
if str(PARSEQ_DIR) not in sys.path:
    sys.path.insert(0, str(PARSEQ_DIR))

print("Using repo zip:", REPO_ZIP)
print("Using repo:", PARSEQ_DIR)
print("Using charset:", PARSEQ_DIR / "configs" / "charset" / f"{CHARSET}.yaml")


Using repo zip: /content/drive/MyDrive/Parseq/parseq_full_data_patch_token_aware_pretrained_transfer_20260527.zip
Using repo: /content/parseq
Using charset: /content/parseq/configs/charset/hybrid_no_val.yaml


## Install Dependencies


In [4]:
import subprocess
import sys

packages = [
    "numpy==1.26.4",
    "opencv-python-headless==4.10.0.84",
    "lmdb",
    "hydra-core>=1.3,<1.4",
    "omegaconf>=2.3,<2.4",
    "pytorch-lightning>=2.2,<3",
    "torchmetrics>=1.3,<2",
    "timm",
    "einops",
    "nltk",
    "fire",
    "rich",
    "imgaug==0.4.0",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PARSEQ_DIR)], check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', '-e', '/content/parseq'], returncode=0)

## Check Environment


In [5]:
import cv2
import hydra
import lmdb
import numpy
import omegaconf
import pytorch_lightning
import timm
import torch

print("NumPy:", numpy.__version__)
print("OpenCV:", cv2.__version__)
print("Torch:", torch.__version__)
print("PyTorch Lightning:", pytorch_lightning.__version__)
print("Hydra:", hydra.__version__)
print("OmegaConf:", omegaconf.__version__)
print("timm:", timm.__version__)

try:
    import imgaug
    print("imgaug:", imgaug.__version__)
except Exception as exc:
    if AUGMENT:
        raise
    print("imgaug unavailable:", exc)

if torch.cuda.is_available():
    ACCELERATOR = "gpu"
    DEVICES = 1
    torch.set_float32_matmul_precision("high")
else:
    ACCELERATOR = "cpu"
    DEVICES = 1

print("ACCELERATOR:", ACCELERATOR)
print("DEVICES:", DEVICES)


NumPy: 1.26.4
OpenCV: 4.11.0
Torch: 2.11.0+cu128
PyTorch Lightning: 2.6.5
Hydra: 1.3.2
OmegaConf: 2.3.0
timm: 1.0.27
imgaug: 0.4.0
ACCELERATOR: gpu
DEVICES: 1


## Copy Dataset


In [6]:
import shutil
import subprocess
from pathlib import Path

if not DATA_SOURCE_DIR.exists():
    raise FileNotFoundError(DATA_SOURCE_DIR)

if WORK_DATA_DIR.exists():
    shutil.rmtree(WORK_DATA_DIR)

WORK_DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
try:
    subprocess.run(["rsync", "-a", "--info=progress2", str(DATA_SOURCE_DIR) + "/", str(WORK_DATA_DIR) + "/"], check=True)
except Exception:
    shutil.copytree(DATA_SOURCE_DIR, WORK_DATA_DIR)

train_root = WORK_DATA_DIR / "train" / "real"
train_dbs = sorted(train_root.rglob("data.mdb")) if train_root.exists() else []
if not train_dbs:
    raise FileNotFoundError(train_root)

print("Training LMDBs:")
for db in train_dbs:
    print(db)


Training LMDBs:
/content/parseq_data/train/real/data.mdb


## Build Training Script


In [7]:
import shlex
from pathlib import Path

RUNS_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    "HYDRA_FULL_ERROR=1 python train.py",
    "  +experiment=parseq-tiny",
    f"  pretrained={PRETRAINED}",
    f"  charset={CHARSET}",
    "  'model.charset_test=${model.charset_train}'",
    f"  data.root_dir={shlex.quote(str(WORK_DATA_DIR))}",
    "  data.train_dir=real",
    "  data.remove_whitespace=false",
    "  data.normalize_unicode=false",
    f"  data.augment={str(AUGMENT).lower()}",
    f"  data.num_workers={NUM_WORKERS}",
    f"  model.batch_size={BATCH_SIZE}",
    f"  model.max_label_length={MAX_LABEL_LENGTH}",
    f"  trainer.accelerator={ACCELERATOR}",
    f"  trainer.devices={DEVICES}",
    f"  trainer.max_epochs={MAX_EPOCHS}",
    f"  '+trainer.default_root_dir={str(RUNS_DIR)}'",
    f"  +trainer.log_every_n_steps={LOG_EVERY_N_STEPS}",
    f"  'hydra.run.dir={str(RUNS_DIR)}/${{now:%Y-%m-%d_%H-%M-%S}}'",
    f"  '+trainer.enable_progress_bar={ENABLE_PROGRESS_BAR}'",
    f"  pretrained_charset={PRETRAINED_CHARSET}",
    f"  pretrained_transfer_vocab_by_token={str(PRETRAINED_TRANSFER_VOCAB_BY_TOKEN).lower()}",
    f"  pretrained_init_new_tokens_from_spelling={str(PRETRAINED_INIT_NEW_TOKENS_FROM_SPELLING).lower()}",
]

if NO_VALIDATION_RUN:
    cmd.extend([
        "  train_without_val=true",
        "  data.allow_missing_val=true",
        "  '+trainer.limit_val_batches=0'",
        "  '+trainer.num_sanity_val_steps=0'",
    ])
else:
    cmd.append("  trainer.val_check_interval=1.0")

script = "#!/usr/bin/env bash\nset -euo pipefail\ncd " + shlex.quote(str(PARSEQ_DIR)) + "\n\n" + " \\\n".join(cmd) + "\n"
TRAIN_SCRIPT = Path("/content/train_parseq_colab.sh")
TRAIN_SCRIPT.write_text(script)
TRAIN_SCRIPT.chmod(0o755)
print(script)


#!/usr/bin/env bash
set -euo pipefail
cd /content/parseq

HYDRA_FULL_ERROR=1 python train.py \
  +experiment=parseq-tiny \
  pretrained=parseq-tiny \
  charset=hybrid_no_val \
  'model.charset_test=${model.charset_train}' \
  data.root_dir=/content/parseq_data \
  data.train_dir=real \
  data.remove_whitespace=false \
  data.normalize_unicode=false \
  data.augment=true \
  data.num_workers=2 \
  model.batch_size=32 \
  model.max_label_length=44 \
  trainer.accelerator=gpu \
  trainer.devices=1 \
  trainer.max_epochs=200 \
  '+trainer.default_root_dir=/content/drive/MyDrive/Parseq/Training/no_val/runs' \
  +trainer.log_every_n_steps=5 \
  'hydra.run.dir=/content/drive/MyDrive/Parseq/Training/no_val/runs/${now:%Y-%m-%d_%H-%M-%S}' \
  '+trainer.enable_progress_bar=true' \
  pretrained_charset=94_full \
  pretrained_transfer_vocab_by_token=true \
  pretrained_init_new_tokens_from_spelling=true \
  train_without_val=true \
  data.allow_missing_val=true \
  '+trainer.limit_val_batches=0' \


## Train


In [ ]:
import subprocess

subprocess.run(["bash", str(TRAIN_SCRIPT)], check=True)
subprocess.run(["bash", "-lc", f'find "{RUNS_DIR}" -name "*.ckpt" -print | sort'], check=False)


## Disconnect


In [ ]:
from google.colab import runtime

runtime.unassign()
